In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GlobalPointer(nn.Module):
    def __init__(self, hidden_size, heads, head_size, RoPE=True):
        super(GlobalPointer, self).__init__()
        self.heads = heads
        self.head_size = head_size
        self.RoPE = RoPE
        # 각 엔티티 타입(heads)별로 Query와 Key를 생성하기 위한 선형 층
        self.dense = nn.Linear(hidden_size, heads * head_size * 2)

    def rotary_position_embedding(self, q, k):
        """RoPE 적용: 상대적 위치 정보를 내적 연산에 주입"""
        batch_size, heads, seq_len, head_size = q.shape
        # 위치 인덱스 생성
        pos = torch.arange(seq_len, device=q.device).type_as(q)
        # 주파수 성분 생성
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_size, 2, device=q.device).float() / head_size))
        sinusoid_inp = torch.einsum("i,j->ij", pos, inv_freq)
        sin, cos = sinusoid_inp.sin(), sinusoid_inp.cos()

        # 짝수/홀수 인덱스 분리하여 회전 변환 적용
        def apply_rope(t):
            t_re, t_im = t[..., 0::2], t[..., 1::2]
            return torch.stack([
                t_re * cos - t_im * sin,
                t_re * sin + t_im * cos
            ], dim=-1).flatten(-2)

        return apply_rope(q), apply_rope(k)

    def forward(self, inputs, mask=None):
        batch_size, seq_len, _ = inputs.shape

        # 1. 선형 변환 및 Query, Key 분리
        inputs = self.dense(inputs)
        inputs = torch.split(inputs, self.head_size * 2, dim=-1)
        inputs = torch.stack(inputs, dim=1) # [batch, heads, seq_len, head_size * 2]
        q, k = inputs[..., :self.head_size], inputs[..., self.head_size:]

        # 2. RoPE 적용 (선택 사항)
        if self.RoPE:
            q, k = self.rotary_position_embedding(q, k)

        # 3. 스팬 점수 계산 (Matrix Multiplication)
        # [batch, heads, seq_len, seq_len]
        logits = torch.matmul(q, k.transpose(-1, -2)) / (self.head_size ** 0.5)

        # 4. 마스킹 (시작점 > 끝점인 경우 및 패딩 제외)
        if mask is not None:
            mask = mask[:, None, None, :] * mask[:, None, :, None]
            logits = logits - (1 - mask) * 1e12

        # 시작 인덱스가 끝 인덱스보다 큰 경우(하삼각 행렬) 제외
        pad_mask = torch.tril(torch.ones_like(logits), -1)
        logits = logits - pad_mask * 1e12

        return logits

def global_pointer_loss(y_true, y_pred):
    """
    논문에서 제안한 멀티라벨 Cross Entropy 손실 함수
    y_true: [batch, heads, seq_len, seq_len], 0 또는 1
    y_pred: [batch, heads, seq_len, seq_len], 모델 출력 (Logits)
    """
    shape = y_pred.shape
    # 원본 행렬을 벡터로 평탄화
    y_true = y_true.view(shape[0] * shape[1], -1)
    y_pred = y_pred.view(shape[0] * shape[1], -1)

    # 논문의 식: log(1 + sum(exp(neg_scores))) + log(1 + sum(exp(-pos_scores)))
    y_pred = (1 - 2 * y_true) * y_pred
    y_pred_neg = y_pred - y_true * 1e12  # 정답인 곳은 아주 작은 값으로
    y_pred_pos = y_pred - (1 - y_true) * 1e12 # 정답이 아닌 곳은 아주 작은 값으로

    zeros = torch.zeros(y_pred.shape[0], 1, device=y_pred.device)
    neg_loss = torch.logsumexp(torch.cat([y_pred_neg, zeros], dim=-1), dim=-1)
    pos_loss = torch.logsumexp(torch.cat([y_pred_pos, zeros], dim=-1), dim=-1)

    return (neg_loss + pos_loss).mean()